## Just some basic EDA and a how-to guide. ##

In [126]:
# %load_ext cudf.pandas # Load CUDF
import re
import os
from pathlib import Path
import kagglehub

# Download latest version

# "magic" command to make plots show up in the notebook
%matplotlib inline 
filepath = Path(kagglehub.dataset_download("wordsforthewise/lending-club"))

print("Path to dataset files:", filepath)

Path to dataset files: /home/vlitoux/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3


Kaggle changed they way they handle compressed files sometime in 2018.  They automatically uncompress them, and put them in a folder with the same name as the file (but all lowercased).  Here is where they are:

In [8]:
folders = os.listdir(filepath)
# Skip .xslx file, if we are ever able to upload it...
# Currently can't upload due to conflicts with other versions of the dataset on Kaggle.
folders = [f for f in folders if 'xlsx' not in f]
folders

['rejected_2007_to_2018q4.csv',
 'rejected_2007_to_2018Q4.csv.gz',
 'accepted_2007_to_2018Q4.csv.gz',
 'accepted_2007_to_2018q4.csv']

In [9]:
accepted_fn = filepath / 'accepted_2007_to_2018q4.csv' / 'accepted_2007_to_2018Q4.csv'

rejected_fn =  filepath / 'rejected_2007_to_2018q4.csv' / 'rejected_2007_to_2018Q4.csv'

accepted_fn.exists(), rejected_fn.exists()

(True, True)

In [ ]:
try:
    import cudf as pd
    BACKEND = "cudf (GPU)"
except Exception:  # ImportError, or CUDA/driver errors on GPU-less machines
    import pandas as pd
    BACKEND = "pandas (CPU)"

print(f"Using {BACKEND}")


acc_df = pd.read_csv(accepted_fn)
# Takes a while to read, because these files are large...give it a minute or so
# acc_df = pd.read_csv(accepted_fn)

# this is a dataset with rejected loans from lendingclub

acc_df

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,<NA>,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,<NA>,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,<NA>,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,<NA>,35000.0,35000.0,35000.0,60 months,14.85,829.9,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,<NA>,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,88985880,<NA>,40000.0,40000.0,40000.0,60 months,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,<NA>,24000.0,24000.0,24000.0,60 months,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0
2260698,88215728,<NA>,14000.0,14000.0,14000.0,60 months,14.49,329.33,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260699,Total amount funded in policy code 1: 1465324575,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

rej_df = pd.read_csv(rejected_fn)
rej_df

,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0
...,...,...,...,...,...,...,...,...,...
27648736,10000.0,2016-12-31,Debt consolidation,590.0,41.26%,441xx,OH,< 1 year,0.0
27648737,10000.0,2016-12-31,moving,NaN,1.48%,207xx,MD,5 years,0.0
27648738,1200.0,2016-12-31,Other,686.0,10.26%,914xx,CA,< 1 year,0.0
27648739,25000.0,2016-12-31,debt_consolidation,NaN,17.71%,880xx,NM,< 1 year,0.0


In [98]:
REJECTED_CONVERT_COLUMNS = {
    "Amount Requested": "loan_amnt",        # requested amount
    "Application Date": "issue_d",          # loan issue month
    "Loan Title": "title",                  # free-text title (not the category "purpose")
    "Risk_Score": "fico_range_low",         # accepted loans store a FICO range (low/high); pick one bound
    "Debt-To-Income Ratio": "dti",
    "Zip Code": "zip_code",
    "State": "addr_state",
    "Employment Length": "emp_length_str",
    "Policy Code": "policy_code",
    "debt_income_ratio": "dti",             # derived duplicate of Debt-To-Income Ratio
    "employment_lenght_years": "emp_length",  # numeric version of Employment Length
}
rej_df = rej_df.rename(columns=REJECTED_CONVERT_COLUMNS)
rej_df


,loan_amnt,issue_d,title,fico_range_low,dti,zip_code,addr_state,emp_length_str,policy_code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0
...,...,...,...,...,...,...,...,...,...
27648736,10000.0,2016-12-31,Debt consolidation,590.0,41.26%,441xx,OH,< 1 year,0.0
27648737,10000.0,2016-12-31,moving,NaN,1.48%,207xx,MD,5 years,0.0
27648738,1200.0,2016-12-31,Other,686.0,10.26%,914xx,CA,< 1 year,0.0
27648739,25000.0,2016-12-31,debt_consolidation,NaN,17.71%,880xx,NM,< 1 year,0.0


In [99]:
def percent_to_float(col: pd.Series):
    return col.str.removesuffix("%").astype("float64")
rej_df["dti"] = percent_to_float(rej_df["dti"])
rej_df

,loan_amnt,issue_d,title,fico_range_low,dti,zip_code,addr_state,emp_length_str,policy_code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10.00,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10.00,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10.00,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43,209xx,MD,< 1 year,0.0
...,...,...,...,...,...,...,...,...,...
27648736,10000.0,2016-12-31,Debt consolidation,590.0,41.26,441xx,OH,< 1 year,0.0
27648737,10000.0,2016-12-31,moving,NaN,1.48,207xx,MD,5 years,0.0
27648738,1200.0,2016-12-31,Other,686.0,10.26,914xx,CA,< 1 year,0.0
27648739,25000.0,2016-12-31,debt_consolidation,NaN,17.71,880xx,NM,< 1 year,0.0


In [101]:

EMPLOYMENT_LENGTH_TO_YEARS = {
    "< 1 year": 1,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 11,
}

def employment_length_to_years(employment_length: pd.Series) -> pd.Series:
    """Convert an 'Employment Length' column to numeric years.

    '< 1 year' -> 1, 'N years' -> N, '10+ years' -> 11.
    NaN (and any unknown value) stays null.
    """
    return employment_length.map(EMPLOYMENT_LENGTH_TO_YEARS)
rej_df["emp_length"] = employment_length_to_years(rej_df["emp_length_str"])
rej_df

,loan_amnt,issue_d,title,fico_range_low,dti,zip_code,addr_state,emp_length_str,policy_code,emp_length
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10.00,481xx,NM,4 years,0.0,4.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10.00,010xx,MA,< 1 year,0.0,1.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10.00,212xx,MD,1 year,0.0,1.0
3,6000.0,2007-05-27,waksman,698.0,38.64,017xx,MA,< 1 year,0.0,1.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43,209xx,MD,< 1 year,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
27648736,10000.0,2016-12-31,Debt consolidation,590.0,41.26,441xx,OH,< 1 year,0.0,1.0
27648737,10000.0,2016-12-31,moving,NaN,1.48,207xx,MD,5 years,0.0,5.0
27648738,1200.0,2016-12-31,Other,686.0,10.26,914xx,CA,< 1 year,0.0,1.0
27648739,25000.0,2016-12-31,debt_consolidation,NaN,17.71,880xx,NM,< 1 year,0.0,1.0


In [102]:
rej_df.dtypes

loan_amnt         float64
issue_d               str
title                 str
fico_range_low    float64
dti               float64
zip_code              str
addr_state            str
emp_length_str        str
policy_code       float64
emp_length        float64
dtype: object

In [103]:
rej_df.describe()

,loan_amnt,fico_range_low,dti,policy_code,emp_length
count,2.764874e+07,9.151111e+06,2.764874e+07,2.764782e+07,2.669739e+07
mean,1.313324e+04,6.281721e+02,1.433401e+02,6.375113e-03,1.589495e+00
std,1.500964e+04,8.993679e+01,1.053916e+04,1.127368e-01,1.741103e+00
min,0.000000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,1.000000e+00
25%,4.800000e+03,5.910000e+02,8.060000e+00,0.000000e+00,1.000000e+00
50%,1.000000e+04,6.370000e+02,1.998000e+01,0.000000e+00,1.000000e+00
75%,2.000000e+04,6.750000e+02,3.661000e+01,0.000000e+00,1.000000e+00
max,1.400000e+06,9.900000e+02,5.000003e+07,2.000000e+00,1.100000e+01


In [104]:
rej_df.dropna()

,loan_amnt,issue_d,title,fico_range_low,dti,zip_code,addr_state,emp_length_str,policy_code,emp_length
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10.00,481xx,NM,4 years,0.0,4.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10.00,010xx,MA,< 1 year,0.0,1.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10.00,212xx,MD,1 year,0.0,1.0
3,6000.0,2007-05-27,waksman,698.0,38.64,017xx,MA,< 1 year,0.0,1.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43,209xx,MD,< 1 year,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
27648733,30000.0,2016-12-31,Debt consolidation,681.0,55.15,726xx,AR,< 1 year,0.0,1.0
27648734,1000.0,2016-12-31,Debt consolidation,531.0,31.31,761xx,TX,< 1 year,0.0,1.0
27648736,10000.0,2016-12-31,Debt consolidation,590.0,41.26,441xx,OH,< 1 year,0.0,1.0
27648738,1200.0,2016-12-31,Other,686.0,10.26,914xx,CA,< 1 year,0.0,1.0


In [105]:
import plotly.express as px

def show_correlation_matrix(df: pd.DataFrame, method: str = "pearson") -> cudf.DataFrame:
    """Compute and display the correlation matrix between the numeric variables of a cuDF DataFrame."""
    numeric_df = df.select_dtypes(include="number")
    corr = numeric_df.corr(method=method)  # computed on GPU, returns a cuDF DataFrame

    fig = px.imshow(
        corr.to_pandas(),                      # plotly needs host data
        text_auto=".2f",                       # annotate each cell
        zmin=-1, zmax=1,
        color_continuous_scale="RdBu_r",       # red = positive, blue = negative
        title=f"{method.capitalize()} correlation matrix",
        aspect="auto",
    )
    fig.update_layout(coloraxis_colorbar=dict(title="corr"))
    fig.show()

    return corr

show_correlation_matrix(rej_df.dropna())

,loan_amnt,fico_range_low,dti,policy_code,emp_length
loan_amnt,1.000000,0.244976,0.001229,0.008700,0.065757
fico_range_low,0.244976,1.000000,-0.001734,0.036847,0.123440
dti,0.001229,-0.001734,1.000000,-0.000281,-0.001799
policy_code,0.008700,0.036847,-0.000281,1.000000,0.106965
emp_length,0.065757,0.123440,-0.001799,0.106965,1.000000


In [76]:
list(rej_df.columns)

['Amount Requested',
 'Application Date',
 'Loan Title',
 'Risk_Score',
 'Debt-To-Income Ratio',
 'Zip Code',
 'State',
 'Employment Length',
 'Policy Code',
 'debt_income_ratio',
 'employment_lenght_years']

In [107]:
acc_df_compare_rej = acc_df[rej_df.select_dtypes(include="number").columns]
acc_df_compare_rej["emp_length"] = employment_length_to_years(acc_df_compare_rej["emp_length"])
acc_df_compare_rej

,loan_amnt,fico_range_low,dti,policy_code,emp_length
0,3600.0,675.0,5.91,1.0,11.0
1,24700.0,715.0,16.06,1.0,11.0
2,20000.0,695.0,10.78,1.0,11.0
3,35000.0,785.0,17.06,1.0,11.0
4,10400.0,695.0,25.37,1.0,3.0
...,...,...,...,...,...
2260696,40000.0,705.0,12.75,1.0,9.0
2260697,24000.0,660.0,18.3,1.0,6.0
2260698,14000.0,660.0,23.36,1.0,11.0
2260699,NaN,NaN,NaN,NaN,NaN


In [109]:
show_correlation_matrix(acc_df_compare_rej.dropna())

,loan_amnt,fico_range_low,dti,policy_code,emp_length
loan_amnt,1.0,0.112711089,0.049126563,NaN,0.082418141
fico_range_low,0.112711089,1.0,-0.034479535,NaN,0.022928432
dti,0.049126563,-0.034479535,1.0,NaN,0.019731042
policy_code,NaN,NaN,NaN,NaN,NaN
emp_length,0.082418141,0.022928432,0.019731042,NaN,1.0


In [122]:
STUDIES_COLUMS = ["loan_amnt", "fico_range_low", "dti", "policy_code", "emp_length"] 
IS_ACCEPTED_COLUMN_NAME = "is_accepted"

rej_df[IS_ACCEPTED_COLUMN_NAME] = 0
acc_df_compare_rej[IS_ACCEPTED_COLUMN_NAME] = 1

In [119]:
rej_df.dtypes

loan_amnt         float64
issue_d               str
title                 str
fico_range_low    float64
dti               float64
zip_code              str
addr_state            str
emp_length_str        str
policy_code       float64
emp_length        float64
is_accepted         int64
dtype: object

In [120]:
acc_df[[IS_ACCEPTED_COLUMN_NAME] + STUDIES_COLUMS].dtypes

is_accepted         int64
loan_amnt         float64
fico_range_low    float64
dti               float64
policy_code       float64
emp_length            str
dtype: object

In [123]:

all_df = pd.concat((rej_df[[IS_ACCEPTED_COLUMN_NAME] + STUDIES_COLUMS],acc_df_compare_rej[[IS_ACCEPTED_COLUMN_NAME] + STUDIES_COLUMS]))
all_df

,is_accepted,loan_amnt,fico_range_low,dti,policy_code,emp_length
0,0,1000.0,693.0,10.0,0.0,4.0
1,0,1000.0,703.0,10.0,0.0,1.0
2,0,11000.0,715.0,10.0,0.0,1.0
3,0,6000.0,698.0,38.64,0.0,1.0
4,0,1500.0,509.0,9.43,0.0,1.0
...,...,...,...,...,...,...
2260696,1,40000.0,705.0,12.75,1.0,9.0
2260697,1,24000.0,660.0,18.3,1.0,6.0
2260698,1,14000.0,660.0,23.36,1.0,11.0
2260699,1,NaN,NaN,NaN,NaN,NaN


In [125]:
show_correlation_matrix(all_df.dropna())

,is_accepted,loan_amnt,fico_range_low,dti,policy_code,emp_length
is_accepted,1.000000,0.096384,0.321427,-0.003707,0.983523,0.632546
loan_amnt,0.096384,1.000000,0.248367,0.000781,0.096244,0.110464
fico_range_low,0.321427,0.248367,1.000000,-0.002811,0.322343,0.265836
dti,-0.003707,0.000781,-0.002811,1.000000,-0.003697,-0.003285
policy_code,0.983523,0.096244,0.322343,-0.003697,1.000000,0.632265
emp_length,0.632546,0.110464,0.265836,-0.003285,0.632265,1.000000


In [56]:
rej_df.shape

(25054319, 9)

In [6]:
[col for col in acc_df.columns if 'fico' in col.lower()]

['fico_range_low',
 'fico_range_high',
 'last_fico_range_high',
 'last_fico_range_low',
 'sec_app_fico_range_low',
 'sec_app_fico_range_high']

In [71]:
# No fico score in rejected loans
[col for col in rej_df.columns if 'fico' in col.lower()]

[]

In [72]:
rej_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25054319 entries, 0 to 25054318
Data columns (total 9 columns):
Amount Requested        float64
Application Date        object
Loan Title              object
Risk_Score              float64
Debt-To-Income Ratio    object
Zip Code                object
State                   object
Employment Length       object
Policy Code             float64
dtypes: float64(3), object(6)
memory usage: 1.7+ GB


In [73]:
acc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2132287 entries, 0 to 2132286
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.4+ GB


In [21]:
# this is how many rows pandas will show by default with methods like pd.dataframe.head()
pd.options.display.max_rows

60

In [22]:
# we want to increase it, because in this case there are a lot of column names
pd.options.display.max_rows = 1000

In [113]:
# the .T is transposing the matrix.
# We do this so the 111 column dataframe is easier to read (easier to scroll down than sideways) 
# .head() shows the first few rows of the data
acc_df.head().T

ValueError: Columns must all have the same dtype

In [60]:
# .tail() shows the last few rows
acc_df.tail().T

,2132282,2132283,2132284,2132285,2132286
id,125893755,125340736,125551208,Total amount funded in policy code 1: 1741781700,Total amount funded in policy code 2: 564202131
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,6000,12000,14000,NaN,NaN
funded_amnt,6000,12000,14000,NaN,NaN
funded_amnt_inv,6000,11975,13975,NaN,NaN
term,36 months,36 months,36 months,NaN,NaN
int_rate,10.91,10.42,13.59,NaN,NaN
installment,196.18,389.58,475.71,NaN,NaN
grade,B,B,C,NaN,NaN
sub_grade,B4,B3,C2,NaN,NaN


In [112]:
# .info() tells us the datatype(int64, `object` is a string)
# and will also tell us the number of non-null (not missing) data points for each column
# because this dataframe is so large, we have to force it to show the datatypes and non-null numbers with the arguments
acc_df.info(verbose=True, null_counts=True)

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 151 columns):
 #   Column                                      Non-Null Count    Dtype
---  ------                                      --------------    -----
 0   id                                          2260701 non-null  str
 1   member_id                                   0 non-null        int8
 2   loan_amnt                                   2260668 non-null  float64
 3   funded_amnt                                 2260668 non-null  float64
 4   funded_amnt_inv                             2260668 non-null  float64
 5   term                                        2260668 non-null  str
 6   int_rate                                    2260668 non-null  float64
 7   installment                                 2260668 non-null  float64
 8   grade                                       2260668 non-null  str
 9   sub_grade                                   2260668 non-null  str
 10  em

# Some common pandas functions

We find ourselves doing the same types of things often in pandas.  For example, masking.  

## Masking dataframes

Maybe we only want to see one column.  We do this like

`acc_df['int_rate']`

We can also subset a dataframe based on some critereon.  Let's look at only the highest interest rates

`acc_df[acc_df['int_rate'] > 20]`

We can take the mean, median, get the standard deviation of columns:

`acc_df['int_rate'].mean()`

In [ ]:
# shows some common summary statistics
# again, transposing with .T to make it easier to read
acc_df.describe().T

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
# "magic" command to make plots show up in the notebook
%matplotlib inline 

In [110]:
acc_df[acc_df['int_rate'] > 20]['int_rate'].mean()
# thats a bit of a complicated statement.  Breaking it down:
# acc_df['int_rate'] > 20 returns a mask: an array of True/False values
# putting this array into acc_df[] returns the dataframe rows where the interest rates are greater than 20

# so this first part: acc_df[acc_df['int_rate'] > 20]
# gives us a dataframe

# we select a column with ['int_rate'] at the end.  Then we get the average value with .mean()

np.float64(23.781285536727616)

## Finding unique values

Numpy has a function for finding the unique values in an array:

np.unique(array)

This is built into pandas:

acc_df['grade'].unique()

shows us the unique values in that column.

In [ ]:
acc_df['grade'].unique()

In [111]:
# selecting only grade A loans:
acc_df[acc_df['grade'] == 'A'].describe().T

ValueError: Columns must all have the same dtype

In [ ]:
acc_df['loan_status'].unique()

In [ ]:
# looking at only defaulted loans:
default_categories = ['Default', 'Charged Off', 'Does not meet the credit policy. Status:Charged Off']
# .isin() is a trick for checking if something is in a list
# it's a pandas-specific function
acc_df[acc_df['loan_status'].isin(default_categories)].describe().T
# check out the average interest rate and dti (debt-to-income)

In [ ]:
# Seaborn is a recently-created Python library for easily making nice-looking plots
# You will have to install it with `conda install seaborn` or `pip install seaborn`, etc

# The docs are here for this function: http://seaborn.pydata.org/generated/seaborn.distplot.html
# Found by Googling 'seaborn histogram'
# Need to drop missing values, otherwise throws an error
# It would be better to impute missing values.
f = sns.distplot(acc_df['dti'].dropna())

In [ ]:
# outliers are screwing up the histogram... remove them
# adapted from http://stackoverflow.com/questions/11686720/is-there-a-numpy-builtin-to-reject-outliers-from-a-list
# we're using interquartile range to determine outliers
def reject_outliers(sr, iq_range=0.5, side='left', return_mask=False):
    """
    Takes an array (or pandas series) and returns an array with outliers excluded, according to the
    interquartile range.
    
    Parameters:
    -----------
    sr: array
        array of numeric values
    iq_range: float
        percent to calculate quartiles by, 0.5 will yield 25% and 75%ile quartiles
    side: string
        if 'left', will return everything below the highest quartile
        if 'right', will return everything above the lowest quartile
        if 'both', will return everything between the high and low quartiles
    """
    pcnt = (1 - iq_range) / 2
    qlow, median, qhigh = sr.dropna().quantile([pcnt, 0.50, 1-pcnt])
    iqr = qhigh - qlow
    if side=='both':
        mask = (sr - median).abs() <= iqr
    elif side=='left':
        mask = (sr - median) <= iqr
    elif side=='right':
        mask = (sr - median) >= iqr
    else:
        print('options for side are left, right, or both')
    
    if return_mask:
        return mask
    
    return sr[mask]

In [ ]:
# sweeeeeeeetttt....
dti_no_outliers = reject_outliers(acc_df['dti'], iq_range=0.85) # arrived at 0.85 via trial and error
f = sns.distplot(dti_no_outliers)
# other types of plot examples:
# http://seaborn.pydata.org/examples/

In [ ]:
# sets the xkcd style if you want to make the plots look funny...may neet to install some fonts
plt.xkcd()

In [ ]:
f = sns.distplot(dti_no_outliers)

In [ ]:
# back to default
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

In [ ]:
f = sns.distplot(dti_no_outliers)

In [ ]:
# other styles:
# http://matplotlib.org/users/style_sheets.html
# this is the R ggplot style, which some people really like
plt.style.use('ggplot')

In [ ]:
# http://seaborn.pydata.org/generated/seaborn.regplot.html
# takes a long time because there are a lot of points, but works for smaller datasets
# f = sns.regplot(data=acc_df, x='dti', y='int_rate', fit_reg=False)
# instead, lets make a heatmap:
# http://stackoverflow.com/questions/2369492/generate-a-heatmap-in-matplotlib-using-a-scatter-data-set
# http://seaborn.pydata.org/generated/seaborn.jointplot.html
mask = reject_outliers(acc_df['dti'], iq_range=0.85, return_mask=True)
f = sns.jointplot(data=acc_df.ix[mask, :], x='dti', y='int_rate', kind='hex', joint_kws=dict(gridsize=50))